# FastAPI Dependency Injection

Dependency Injection (DI) in FastAPI allows you to declare shared logic, database sessions, authentication, and more as reusable dependencies. FastAPI resolves them automatically at request time.

In [ ]:
# Install if needed
# !pip install fastapi httpx

from fastapi import FastAPI, Depends, Query
from fastapi.testclient import TestClient

app = FastAPI()

# --- 1. Simple function dependency for pagination ---
def pagination_params(
    skip: int = Query(0, ge=0, description="Number of items to skip"),
    limit: int = Query(10, ge=1, le=100, description="Max items to return")
):
    return {"skip": skip, "limit": limit}

@app.get("/items")
def list_items(pagination: dict = Depends(pagination_params)):
    fake_db = [f"item_{i}" for i in range(100)]
    s, l = pagination["skip"], pagination["limit"]
    return {"items": fake_db[s:s+l], "skip": s, "limit": l}

client = TestClient(app)

# Default pagination
r = client.get("/items")
print("Default:", r.json())

# Custom pagination
r = client.get("/items?skip=5&limit=3")
print("Custom:", r.json())

In [ ]:
# --- 2. Class-based dependency ---
from fastapi import FastAPI, Depends
from fastapi.testclient import TestClient

app2 = FastAPI()

class CommonQueryParams:
    def __init__(self, q: str = None, skip: int = 0, limit: int = 10):
        self.q = q
        self.skip = skip
        self.limit = limit

@app2.get("/search")
def search(params: CommonQueryParams = Depends()):
    results = [f"result_{i}" for i in range(50)]
    if params.q:
        results = [r for r in results if params.q in r]
    return {
        "query": params.q,
        "results": results[params.skip:params.skip + params.limit]
    }

client2 = TestClient(app2)

r = client2.get("/search?q=result_1&limit=5")
print("Class-based dep:", r.json())

In [ ]:
# --- 3. Dependency with yield (simulated DB session) ---
from fastapi import FastAPI, Depends
from fastapi.testclient import TestClient
from typing import Generator

app3 = FastAPI()

# Simulated in-memory "database"
FAKE_DB = {"users": {1: {"id": 1, "name": "Alice"}, 2: {"id": 2, "name": "Bob"}}}

def get_db() -> Generator:
    """Yield a DB session (dict here), then clean up."""
    db = FAKE_DB  # In real code: SessionLocal()
    try:
        print("  [DB] Session opened")
        yield db
    finally:
        print("  [DB] Session closed")

@app3.get("/users/{user_id}")
def get_user(user_id: int, db: dict = Depends(get_db)):
    user = db["users"].get(user_id)
    if not user:
        from fastapi import HTTPException
        raise HTTPException(status_code=404, detail="User not found")
    return user

client3 = TestClient(app3)

print("Fetching user 1:")
r = client3.get("/users/1")
print("  Response:", r.json())

print("Fetching user 99 (not found):")
r = client3.get("/users/99")
print("  Response:", r.status_code, r.json())

In [ ]:
# --- 4. Sub-dependencies and caching demo ---
from fastapi import FastAPI, Depends, Header, HTTPException
from fastapi.testclient import TestClient

app4 = FastAPI()

call_count = {"verify_token": 0}

def verify_token(x_token: str = Header(...)):
    call_count["verify_token"] += 1
    print(f"  verify_token called (count={call_count['verify_token']})")
    if x_token != "secret-token":
        raise HTTPException(status_code=403, detail="Invalid token")
    return x_token

def get_current_user(token: str = Depends(verify_token)):
    # Sub-dependency: depends on verify_token
    return {"user": "alice", "token": token}

@app4.get("/profile")
def profile(user: dict = Depends(get_current_user)):
    return user

@app4.get("/dashboard")
def dashboard(
    user: dict = Depends(get_current_user),
    token: str = Depends(verify_token)  # Same dep — FastAPI caches it per request!
):
    return {"user": user, "cached_token": token}

client4 = TestClient(app4)

# Valid token
call_count["verify_token"] = 0
r = client4.get("/dashboard", headers={"x-token": "secret-token"})
print("Dashboard:", r.json())
print(f"verify_token called {call_count['verify_token']} time(s) — FastAPI caches deps per request")

# Invalid token
r = client4.get("/profile", headers={"x-token": "wrong"})
print("Invalid token:", r.status_code, r.json())

In [ ]:
# --- 5. Global dependency example ---
from fastapi import FastAPI, Depends, Header, HTTPException
from fastapi.testclient import TestClient

def verify_api_key(x_api_key: str = Header(...)):
    if x_api_key != "my-api-key":
        raise HTTPException(status_code=403, detail="Forbidden")

# Apply dependency globally to ALL routes
app5 = FastAPI(dependencies=[Depends(verify_api_key)])

@app5.get("/public-data")
def public_data():
    return {"data": "This still requires the global API key"}

@app5.get("/private-data")
def private_data():
    return {"secret": 42}

client5 = TestClient(app5)

# Without key
r = client5.get("/public-data")
print("No key:", r.status_code, r.json())

# With wrong key
r = client5.get("/public-data", headers={"x-api-key": "bad"})
print("Wrong key:", r.status_code, r.json())

# With correct key
r = client5.get("/public-data", headers={"x-api-key": "my-api-key"})
print("Correct key:", r.json())

r = client5.get("/private-data", headers={"x-api-key": "my-api-key"})
print("Private data:", r.json())